# Day 43：多智能体协作系统（Supervisor 模式：Researcher + Analyst + Writer）

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第四阶段（Day 43）！今天我们将实战演示如何通过 `SupervisorOrchestrator` 统筹 `ResearcherAgent`、`AnalystAgent` 与 `WriterAgent` 完成多智能体协同研究，并验证 `agent_handoffs` 全链路追踪与 `max_rounds` 死循环熔断机制。

## 1. 环境初始化与导入 `project_iterations/day_43` 模块
定位仓库根目录并将 `project_iterations/day_43` 加入 `sys.path`。

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / 'project_iterations').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day43_dir = repo_root / 'project_iterations' / 'day_43'
if str(day43_dir) not in sys.path:
    sys.path.insert(0, str(day43_dir))

from zhishu_agent import (
    FINISH,
    AgentWorker,
    AnalystAgent,
    ResearcherAgent,
    SupervisorOrchestrator,
    WorkerResult,
    WriterAgent,
)

print('Day 43 模块加载成功:', day43_dir)

Day 43 模块加载成功: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_43


## 2. 默认 Supervisor 协作流：Researcher -> Analyst -> Writer
使用默认路由策略执行研究任务，查看 `agent_handoffs` 交接轨迹与最终合成研报。

In [2]:
orchestrator = SupervisorOrchestrator(max_rounds=6)
result = orchestrator.run('企业级多智能体系统的 Supervisor 编排与治理')

assert result.status == 'completed'
assert result.rounds_used == 3

print(f'状态: {result.status} | 使用轮次: {result.rounds_used}/{result.max_rounds}')
for rec in result.agent_handoffs:
    print(f'  Round {rec.round_num}: {rec.from_agent} -> {rec.to_agent} | {rec.summary}')

print('\n--- 最终研报预览 ---')
print(result.final_report)

状态: completed | 使用轮次: 3/6
  Round 1: supervisor -> researcher | 【调研结论】
  Round 2: supervisor -> analyst | 【分析洞察】
  Round 3: supervisor -> writer | # 企业级多智能体系统的 Supervisor 编排与治理 — 多智能体联合研究报告

--- 最终研报预览 ---
# 企业级多智能体系统的 Supervisor 编排与治理 — 多智能体联合研究报告

## 1. 背景与核心事实（Researcher）
1. [事实 1] 企业级多智能体系统的 Supervisor 编排与治理 在企业生产环境中通常面临任务拆解、工具调用可靠性与上下文隔离三大挑战。
2. [事实 2] 采用 Supervisor 集中式路由模式可将单 Agent 提示词复杂度降低约 40%，并支持专职能力独立演进。
3. [事实 3] 生产级多智能体系统必须设置 max_rounds 熔断阈值与 handoff 审计日志以防止死循环。

## 2. 深度分析与关键洞察（Analyst）
- 针对「企业级多智能体系统的 Supervisor 编排与治理」，共汇总 3 条一手调研事实，角色专业化分工收益显著。
- 集中式 Supervisor 相比网状 Peer-to-Peer 通信更易实现权限审计与确定性终止。

## 3. 风险评估与治理建议
- 若路由策略缺少完成态判定或 max_rounds 上限，Worker 间可能发生乒乓式循环交接。
- 长链路上下文透传需控制信息冗余，避免干扰下游撰稿质量。

## 4. 关键指标摘要
- 调研事实数: 3
- 编排模式: Supervisor-Worker



## 3. `max_rounds` 熔断防护：防止 Worker 乒乓死循环
构造一个永不返回 `FINISH` 的自定义路由，验证到达 `max_rounds` 上限后安全熔断。

In [3]:
def ping_pong_router(ctx, handoffs, available):
    target = 'analyst' if len(handoffs) % 2 == 1 else 'researcher'
    return target, f'第 {len(handoffs) + 1} 轮循环验证'

guarded = SupervisorOrchestrator(max_rounds=4, router_fn=ping_pong_router)
loop_res = guarded.run('死循环熔断演练')

assert loop_res.status == 'max_rounds_reached'
assert loop_res.rounds_used == 4
print('熔断状态:', loop_res.status)
print('交接序列:', [h.to_agent for h in loop_res.agent_handoffs])

熔断状态: max_rounds_reached
交接序列: ['researcher', 'analyst', 'researcher', 'analyst']


## 4. 横向扩展：新增 `ReviewerAgent` 质量门禁
继承 `AgentWorker` 实现第 4 个专职角色 `ReviewerAgent`，并接入自定义四阶段路由。

In [4]:
class ReviewerAgent(AgentWorker):
    def __init__(self) -> None:
        super().__init__('reviewer', 'Quality Reviewer', '审核研报结构完整性')

    def execute(self, task: str, context: dict) -> WorkerResult:
        report = str(context.get('report', ''))
        approved = ('核心事实' in report) and ('风险评估' in report)
        return WorkerResult(
            agent_name=self.name,
            role=self.role,
            output=f'质量审核: {approved}',
            artifacts={'review_approved': approved},
        )

def four_stage_router(ctx, handoffs, available):
    if 'findings' not in ctx:
        return 'researcher', '收集事实'
    if 'insights' not in ctx:
        return 'analyst', '分析洞察'
    if 'report' not in ctx:
        return 'writer', '撰写报告'
    if 'review_approved' not in ctx:
        return 'reviewer', '质量审核'
    return FINISH, '审核通过'

team = SupervisorOrchestrator(
    workers=[ResearcherAgent(), AnalystAgent(), WriterAgent(), ReviewerAgent()],
    max_rounds=6,
    router_fn=four_stage_router,
)
rev_res = team.run('企业级 AI 网关选型')
assert rev_res.status == 'completed'
assert rev_res.shared_context['review_approved'] is True
print('四阶段流转:', [h.to_agent for h in rev_res.agent_handoffs])
print('审核结果:', rev_res.worker_outputs['reviewer'])

四阶段流转: ['researcher', 'analyst', 'writer', 'reviewer']
审核结果: 质量审核: True
